In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Point Python to your src folder so it can find config.py
# Robust src path — works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

print("Packages loaded successfully")
print(f"Raw data path : {RAW_PATH}")
print(f"Output path   : {OUT_PATH}")

Packages loaded successfully
Raw data path : data/raw/manual_validation.xlsx
Output path   : data/outputs/TT/


In [2]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / RAW_PATH

df = pd.read_excel(RAW_FILE, sheet_name=RAW_SHEET)

print("=" * 50)
print("SHAPE")
print(f"{df.shape[0]} rows  {df.shape[1]} columns")

print("\nCOLUMNS")
for col in df.columns:
    print(col)

print("\nDATATYPES")
print(df.dtypes.to_string())

SHAPE
61390 rows  15 columns

COLUMNS
year
month
Portal
Brand Code Text
Brand(Masked)
Division
Material Number
Masked SKU
Range
Masked Range
Size
Final Channel
Final Sub Channel
qty
net_sales

DATATYPES
year                   int64
month                  int64
Portal                 int64
Brand Code Text       object
Brand(Masked)         object
Division              object
Material Number       object
Masked SKU            object
Range                 object
Masked Range          object
Size                  object
Final Channel         object
Final Sub Channel     object
qty                    int64
net_sales            float64


In [3]:
print("MISSING VALUES")
print(df.isnull().sum().to_string())

print("\nNEGATIVE OR ZERO QTY")
print(f"  qty <= 0  : {(df['qty'] <= 0).sum()} rows")
print(f"  net_sales <= 0 : {(df['net_sales'] <= 0).sum()} rows")

print("\nYEAR DISTRIBUTION")
print(df['year'].value_counts().sort_index().to_string())

print("\nFINAL CHANNEL DISTRIBUTION")
print(df['Final Channel'].value_counts().to_string())

MISSING VALUES
year                 0
month                0
Portal               0
Brand Code Text      0
Brand(Masked)        0
Division             0
Material Number      0
Masked SKU           0
Range                0
Masked Range         0
Size                 0
Final Channel        0
Final Sub Channel    0
qty                  0
net_sales            0

NEGATIVE OR ZERO QTY
  qty <= 0  : 0 rows
  net_sales <= 0 : 0 rows

YEAR DISTRIBUTION
year
2023    17485
2024    21694
2025    22211

FINAL CHANNEL DISTRIBUTION
Final Channel
TT    61390


In [4]:
print("FULL YEAR DISTRIBUTION")
print(df['year'].value_counts().sort_index().to_string())

print("\nDIVISION DISTRIBUTION")
print(df['Division'].value_counts().to_string())

print("\nSIZE DISTRIBUTION")
print(df['Size'].value_counts().to_string())

print("\nSEGMENT INVENTORY (Division x Portal x Size)")
print("How many rows each segment has:\n")
inventory = (df.groupby(['Division', 'Portal', 'Size'])
               .size()
               .reset_index(name='row_count')
               .sort_values('row_count', ascending=False))
print(inventory.to_string(index=False))
print(f"\nTotal segments: {len(inventory)}")

FULL YEAR DISTRIBUTION
year
2023    17485
2024    21694
2025    22211

DIVISION DISTRIBUTION


Division
HL    22917
SL    16314
BP    16157
DF     5410
BS      592

SIZE DISTRIBUTION
Size
Single    16749
MEDIUM    10830
CABIN     10610
LARGE     10359
SO3        4714
DFT        4205
SO2        2716
DF         1205
SO3           2

SEGMENT INVENTORY (Division x Portal x Size)
How many rows each segment has:



Division  Portal   Size  row_count
      BP       1 Single      16157
      HL       1 MEDIUM       6792
      HL       1  CABIN       5958
      SL       1  LARGE       5564
      HL       1  LARGE       4795
      SL       1  CABIN       4652
      DF       1    DFT       4205
      SL       1 MEDIUM       4038
      HL       1    SO3       3422
      HL       1    SO2       1948
      SL       1    SO3       1292
      DF       1     DF       1205
      SL       1    SO2        768
      BS       1 Single        592
      HL       1   SO3           2

Total segments: 15


In [5]:
# Strip whitespace from every string column — fixes the SO3 / 'SO3 ' problem
for col in ['Division', 'Size', 'Final Channel'] + ([RANGE_COL] if RANGE_COL in df.columns else []):
    df[col] = df[col].astype(str).str.strip()

# Apply business filters
df_clean = df[~df['year'].isin(IGNORE_YEARS)].copy()
df_clean = df_clean[df_clean['Final Channel'] == CHANNEL].copy()

# Confirm the phantom segment is gone
print("AFTER CLEANING")
print(f"Rows remaining : {len(df_clean):,}")
print(f"Years remaining: {sorted(df_clean['year'].unique())}")

print("\nSEGMENT INVENTORY AFTER CLEANING")
inventory_clean = (df_clean.groupby(['Division', 'Portal', 'Size'])
                            .size()
                            .reset_index(name='row_count')
                            .sort_values(['Division', 'Size']))
print(inventory_clean.to_string(index=False))
print(f"\nTotal segments: {len(inventory_clean)}")

# Check SO3 is now clean
print("\nSO3 check (should be exactly one entry):")
print(inventory_clean[inventory_clean['Size'].str.contains('SO3')])

AFTER CLEANING
Rows remaining : 43,905
Years remaining: [np.int64(2024), np.int64(2025)]

SEGMENT INVENTORY AFTER CLEANING
Division  Portal   Size  row_count
      BP       1 Single      11243
      BS       1 Single        372
      DF       1     DF       1080
      DF       1    DFT       2956
      HL       1  CABIN       4290
      HL       1  LARGE       3433
      HL       1 MEDIUM       5208
      HL       1    SO2       1573
      HL       1    SO3       2939
      SL       1  CABIN       3071
      SL       1  LARGE       3594
      SL       1 MEDIUM       2802
      SL       1    SO2        542
      SL       1    SO3        802

Total segments: 14

SO3 check (should be exactly one entry):
   Division  Portal Size  row_count
8        HL       1  SO3       2939
13       SL       1  SO3        802


In [6]:
df_clean['sale_date'] = pd.to_datetime(
    dict(year=df_clean['year'], month=df_clean['month'], day=1)
)

print("DATE RANGE")
print(f"  From : {df_clean['sale_date'].min().strftime('%b %Y')}")
print(f"  To   : {df_clean['sale_date'].max().strftime('%b %Y')}")
print(f"  Months: {df_clean['sale_date'].nunique()}")

print("\nQTY AND NET_SALES SANITY CHECK")
print(f"  Total qty sold    : {df_clean['qty'].sum():,.0f} units")
print(f"  Total net sales   : ₹{df_clean['net_sales'].sum():,.0f}")
print(f"  Overall avg ASP   : ₹{(df_clean['net_sales'].sum() / df_clean['qty'].sum()):,.0f}")

print("\nQTY PER DIVISION (2024+2025 combined)")
div_summary = (df_clean.groupby('Division')
                       .agg(qty=('qty','sum'), net_sales=('net_sales','sum'))
                       .assign(ASP=lambda x: x['net_sales']/x['qty'])
                       .sort_values('qty', ascending=False))
div_summary['qty']       = div_summary['qty'].map('{:,.0f}'.format)
div_summary['net_sales'] = div_summary['net_sales'].map('₹{:,.0f}'.format)
div_summary['ASP']       = div_summary['ASP'].map('₹{:,.0f}'.format)
print(div_summary.to_string())

# Save clean dataframe for notebook 02
os.makedirs(OUT_PATH, exist_ok=True)
df_clean.to_csv(f'{OUT_PATH}01_clean_sales.csv', index=False)
print(f"\nSaved: {OUT_PATH}01_clean_sales.csv")

DATE RANGE
  From : Jan 2024
  To   : Dec 2025
  Months: 24

QTY AND NET_SALES SANITY CHECK
  Total qty sold    : 4,620,619 units
  Total net sales   : ₹9,670,635,150
  Overall avg ASP   : ₹2,093

QTY PER DIVISION (2024+2025 combined)
                qty       net_sales     ASP
Division                                   
HL        2,019,400  ₹6,334,106,727  ₹3,137
BP        1,099,961    ₹979,275,611    ₹890
DF          848,587    ₹699,224,329    ₹824
SL          641,694  ₹1,636,685,545  ₹2,551
BS           10,977     ₹21,342,938  ₹1,944



Saved: data/outputs/TT/01_clean_sales.csv
